# Data leakage

**Leakage** is information reaching a model during training, or during its evaluation, that it won't have when it is used. The result is always the same: a score better than the model deserves, and a model that disappoints in production. Leakage rarely breaks anything visibly; the code runs, the numbers look good, and that is what makes it dangerous.

Four short demonstrations follow, each with the leaky score next to the honest one. A fifth kind, choosing a model by its test score, was measured in [Validation and model selection](validation_and_model_selection.ipynb#choosing-on-the-test-set).

## Learning from the validation rows

Any step that learns from data, a scaler, an imputer, a selection of features, must learn from the training rows only ([the test set stands for production](overfitting_and_the_split.ipynb#the-test-set-stands-for-production)). For a scaler, learning from all the rows changes the result very little, which is why the mistake goes unnoticed. For feature selection it can change everything.

An extreme case: 100 rows with 10,000 features of pure random noise, and a target that is random noise too. No model can predict it; the honest R² is about 0. Selecting the 20 features most correlated with the target *before* cross-validating, using every row:

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(100, 10_000))
y_noise = rng.normal(size=100)
folds = KFold(n_splits=5, shuffle=True, random_state=0)

selected_with_every_row = SelectKBest(f_regression, k=20).fit_transform(X_noise, y_noise)
cross_val_score(LinearRegression(), selected_with_every_row, y_noise, cv=folds, scoring="r2").mean()

np.float64(0.6082550151604569)

And with the selection inside the pipeline, so that each round selects its features from its own training folds:

In [ ]:
selection_inside = make_pipeline(SelectKBest(f_regression, k=20), LinearRegression())
cross_val_score(selection_inside, X_noise, y_noise, cv=folds, scoring="r2").mean()

np.float64(-0.0306497038602491)

An R² of 0.61 on data that contains no information at all, against about 0 when each round selects its own features. Among 10,000 random features, some happen to be correlated with this particular random target. Choosing them with every row, validation rows included, picks exactly the ones that happen to fit the validation rows as well. The model has learned nothing; the evaluation has been fooled.

## A feature that contains the target

A column can be in the table and still be unavailable when the prediction is made: it was computed from the target, or recorded after the event being predicted. Suppose the California districts came with the property tax their owners pay, which is computed from the value of the house, about 1% of it, with some variation. A model trained with it:

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_with_tax = X.assign(property_tax=0.01 * y * rng.normal(1, 0.05, len(y)))
X_train, X_test, y_train, y_test = train_test_split(X_with_tax, y, test_size=0.2, random_state=42)

with_tax = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
without_tax = HistGradientBoostingRegressor(random_state=42).fit(X_train.drop(columns="property_tax"), y_train)
{
    "with the tax": mean_absolute_error(y_test, with_tax.predict(X_test)),
    "without it": mean_absolute_error(y_test, without_tax.predict(X_test.drop(columns="property_tax"))),
}

{'with the tax': 7738.343639895265, 'without it': 31105.521289073844}

With the tax, the model misses by about \$7,700; without it, by about \$31,100. The tax is 1% of the value with a little variation, so the column is almost the answer: the model only has to read it back. But a house is valued before a tax has been assessed on that value: in production the column wouldn't exist, and the model, which learned to rely on it, would have nothing to fall back on. The same happens whenever a column is computed from the target (a price per square metre, when the price is what is predicted) or recorded after the outcome (a treatment given after a diagnosis, a refund issued after a fraud).

## The same example on both sides

When the same entity appears in several rows, a patient visited twice, a house sold three times, a message sent many times, a random split puts copies of it on both sides, and the test partly measures whether the model remembers it. Suppose each of 3,000 districts appears three times, with values that differ by a few percent, as repeated listings of the same place would. Random folds against folds that keep all the copies of a district together (`GroupKFold`):

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold

districts = housing.sample(3000, random_state=0).reset_index(drop=True)
copies = pd.concat([districts.assign(district=districts.index)] * 3, ignore_index=True)
copies["MedHouseVal"] *= rng.normal(1, 0.03, len(copies))
X_copies = copies.drop(columns=["MedHouseVal", "district"])
y_copies = 100_000 * copies["MedHouseVal"]
forest = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
{
    "random folds": -cross_val_score(forest, X_copies, y_copies, cv=KFold(5, shuffle=True, random_state=0),
                                     scoring="neg_mean_absolute_error").mean(),
    "folds by district": -cross_val_score(forest, X_copies, y_copies, cv=GroupKFold(5), groups=copies["district"],
                                          scoring="neg_mean_absolute_error").mean(),
}

{'random folds': np.float64(11679.135183869794),
 'folds by district': np.float64(40264.298905102325)}

With random folds the forest seems to miss by about \$11,700; with the copies of each district kept together, by about \$40,300, more than three times as much. In random folds, almost every validation district has a copy in the training folds, and the forest finds it and repeats its value. A new district, the kind production will bring, has no copy anywhere. The honest error is also higher than the forest's on the full data (about \$33,600 in [Validation and model selection](validation_and_model_selection.ipynb)), because these folds only have 3,000 distinct districts to learn from.

## Information from the future

When a model will predict the future, the past is all it will have. A random split hands it rows from after the ones it is tested on. Five years of simulated sales of flats, with prices that rise about 1% a month: the same model, tested once on a random 20% of the sales, and once on the last 20% in time, after training on everything before:

In [ ]:
n_sales = 3000
month = np.sort(rng.uniform(0, 60, n_sales))
size = rng.uniform(50, 150, n_sales)
price = 2000 * size * (1 + 0.01 * month) * rng.normal(1, 0.08, n_sales)
sales = pd.DataFrame({"size_m2": size, "month": month, "price": price})

model = HistGradientBoostingRegressor(random_state=0)
random_train, random_test = train_test_split(sales, test_size=0.2, random_state=0)
cut = int(0.8 * n_sales)
past, future = sales.iloc[:cut], sales.iloc[cut:]
{
    "random 20%": mean_absolute_error(random_test["price"], model.fit(random_train[["size_m2", "month"]], random_train["price"])
                                      .predict(random_test[["size_m2", "month"]])),
    "last 20% in time": mean_absolute_error(future["price"], model.fit(past[["size_m2", "month"]], past["price"])
                                            .predict(future[["size_m2", "month"]])),
}

{'random 20%': 16899.887452767074, 'last 20% in time': 22304.700528280682}

Tested on a random 20% of the sales, the model misses by about \$16,900; tested on the last year, by about \$22,300, some 30% more. With random rows, every test month has training sales just before and after it, so the model never has to look beyond the prices it has seen. On the last year it has to predict prices higher than any its training rows show for a flat of that size, and trees can't extrapolate: they predict, at most, the prices of the latest months they saw. A random split of data that drifts over time measures a model that has seen the future; [King County: temporal leakage](https://github.com/avidaldo/king-county/blob/master/03-temporal_leakage.ipynb) shows the same on real house sales.

## Preventing it

| Kind | What leaks | How it is prevented |
| --- | --- | --- |
| Learning from the validation or test rows | their statistics, through a scaler, an imputer, a selection | every step that learns goes inside a pipeline, fitted on the training rows only |
| A feature that contains the target | the answer, through a column computed from it or recorded after it | ask of every column: would it exist at the moment of the prediction? |
| The same example on both sides | memory of specific rows | split by entity (`GroupKFold`), after removing exact duplicates |
| Information from the future | later rows | split by time (`TimeSeriesSplit`, or train on the past and test on the most recent) |
| Choosing on the test set | the test score, through the decision | a validation set or cross-validation for every decision; the test set once |

Suspicion is the best detector: a score much better than expected, or a single feature far more important than the rest, deserves a check before a celebration.